# Part 3: Tile Order Hardness Analysis

This notebook reuses the configured Part 1 CNN baseline results and saved tile permutations to compute tile permutation hardness metrics without retraining.

In [1]:
from pathlib import Path
import importlib
import json
import os
import sys


In [2]:
from pathlib import Path
import importlib
import importlib.util

_REQUIRED_PROJECT_FILES = (
    Path('src/__init__.py'),
    Path('src/utils/notebook_setup.py'),
    Path('src/evaluation/experiment_results.py'),
)

_CANDIDATE_PROJECT_ROOTS = (
    Path('/content/drive/MyDrive/MLDS_Final_Project'),
    Path('/content/MLDS_Final_Project'),
    Path('/content/drive/MyDrive/Colab Notebooks/MLDS_Final_Project'),
)


def _safe_resolve(path):
    try:
        return Path(path).resolve()
    except OSError:
        return None


def _safe_exists(path):
    try:
        return Path(path).exists()
    except OSError:
        return False


def _safe_cwd():
    try:
        return Path.cwd().resolve()
    except OSError:
        fallback = Path('/content')
        return fallback if _safe_exists(fallback) else Path.home()


def _safe_glob(path, pattern):
    try:
        return list(path.glob(pattern))
    except OSError:
        return []


def _path_looks_like_project_root(path):
    try:
        return all((path / required).is_file() for required in _REQUIRED_PROJECT_FILES)
    except OSError:
        return False


def _candidate_project_roots():
    current = _safe_cwd()
    candidates = [current, *current.parents, *_CANDIDATE_PROJECT_ROOTS]
    my_drive = Path('/content/drive/MyDrive')
    if _safe_exists(my_drive):
        candidates.extend(_safe_glob(my_drive, 'MLDS_Final_Project'))
        candidates.extend(_safe_glob(my_drive, '*/MLDS_Final_Project'))
    return candidates


def _find_project_root():
    seen = set()
    for candidate in _candidate_project_roots():
        candidate = _safe_resolve(candidate)
        if candidate is None or candidate in seen:
            continue
        seen.add(candidate)
        if _path_looks_like_project_root(candidate):
            return candidate
    return None


def _mount_colab_drive_if_available():
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        return
    drive.mount('/content/drive', force_remount=True)


_PROJECT_ROOT = _find_project_root()
if _PROJECT_ROOT is None:
    _mount_colab_drive_if_available()
    _PROJECT_ROOT = _find_project_root()

if _PROJECT_ROOT is None:
    raise ModuleNotFoundError(
        'Could not find the full MLDS_Final_Project repo. Expected '
        'src/__init__.py, src/utils/notebook_setup.py, and '
        'src/evaluation/experiment_results.py. In Colab, upload or clone '
        'the full repo, or place it at /content/drive/MyDrive/MLDS_Final_Project.'
    )

_COLAB_UTILS_PATH = _PROJECT_ROOT / 'src' / 'utils' / 'colab.py'
_COLAB_SPEC = importlib.util.spec_from_file_location('_mlds_colab_bootstrap', _COLAB_UTILS_PATH)
if _COLAB_SPEC is None or _COLAB_SPEC.loader is None:
    raise ImportError(f'Could not load Colab bootstrap helpers from {_COLAB_UTILS_PATH}')
_colab_bootstrap = importlib.util.module_from_spec(_COLAB_SPEC)
_COLAB_SPEC.loader.exec_module(_colab_bootstrap)

ROOT = _colab_bootstrap.bootstrap_notebook_runtime(_PROJECT_ROOT, force_remount=False)
notebook_setup = importlib.import_module('src.utils.notebook_setup')
ROOT


Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13
Package versions:
  numpy: 2.0.2
  pandas: 2.2.2
  scipy: 1.16.3
  scikit-learn: 1.6.1
  torch: 2.11.0+cu128
  torchvision: 0.26.0+cu128
  timm: 1.0.27
  pyyaml: 6.0.3
torch.cuda.is_available(): True
CUDA device: NVIDIA L4
Project root: /content/drive/MyDrive/MLDS_Final_Project


PosixPath('/content/drive/MyDrive/MLDS_Final_Project')

In [3]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)


In [4]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage

import src.evaluation.tile_permutation_difficulty as tile_permutation_difficulty
from src.preprocessing.image_transforms import PILToFloatTensor, make_tile_compatible_image_size
from src.preprocessing.tile_permutations import tile_permutation_from_jsonable
from src.preprocessing.tile_transforms import apply_tile_permutation
import src.utils.colab as colab_module

colab_module = importlib.reload(colab_module)
import src.utils.config as config_module

config_module = importlib.reload(config_module)
tile_permutation_difficulty = importlib.reload(tile_permutation_difficulty)
compute_adjacency_destruction_hardness = tile_permutation_difficulty.compute_adjacency_destruction_hardness
compute_combined_hardness = tile_permutation_difficulty.compute_combined_hardness
compute_spatial_permutation_entropy = tile_permutation_difficulty.compute_spatial_permutation_entropy
compute_global_displacement = tile_permutation_difficulty.compute_global_displacement
Part3ExperimentConfig = config_module.Part3ExperimentConfig
load_part1_model_results = experiment_results.load_part1_model_results
load_part3_results = experiment_results.load_part3_results
load_or_build_part1_tile_permutations = experiment_results.load_or_build_part1_tile_permutations
part3_output_paths = experiment_results.part3_output_paths
load_experiment_samples = experiment_results.load_experiment_samples
stage_configured_colab_data_dir = experiment_results.stage_configured_colab_data_dir
run_part3_hardness_analysis = experiment_results.run_part3_hardness_analysis



## Configuration
Define Part 3 settings in `Part3ExperimentConfig`, reusing the Part 1 model/data/tile permutation outputs.


In [5]:
part3_setup = notebook_setup.setup_part3_config()
config = part3_setup.config
config.stage_colab_data_to_local_disk = True  # Set False on Colab to read directly from Drive instead of copying to /content.
part3_setup.summary['stage_colab_data_to_local_disk'] = config.stage_colab_data_to_local_disk
part3_setup.summary['colab_local_data_dir'] = config.colab_local_data_dir
results_dir = part3_setup.results_dir
figures_dir = part3_setup.figures_dir
part1_results_csv = part3_setup.part1_results_csv
tile_permutation_csv = part3_setup.tile_permutation_csv
output_paths = part3_setup.output_paths
part3_setup.summary


Running on Google Colab, adjusting configs...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,part,config_name,model_name,tiles_per_side_values,num_tile_permutations,tile_permutation_names,grid_x_permutation_records,seed,batch_size,num_workers,use_amp,stage_colab_data_to_local_disk,colab_local_data_dir,weight_adj,weight_entropy,weight_dist
0,part3,part3_hardness_analysis,resnet18,"[1, 4, 7, 10]",3,"[easy, medium, hard]",12,42,64,4,True,True,/content/MLDS_Final_Project/data/dogs-vs-cats/...,0.5,0.3,0.2


In [6]:
print(config)


Part3ExperimentConfig(part='part3', config_name='part3_hardness_analysis', device='auto', deterministic=False, root_dir='/content/drive/MyDrive/MLDS_Final_Project', data_dir='/content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train', outputs_dir='/content/drive/MyDrive/MLDS_Final_Project/outputs', results_dir='/content/drive/MyDrive/MLDS_Final_Project/outputs/results', figures_dir='/content/drive/MyDrive/MLDS_Final_Project/outputs/figures', sample_data=False, sample_limit=256, val_fraction=0.2, test_fraction=0.0, image_size=224, batch_size=64, num_workers=4, num_classes=2, model_names=['resnet18'], pretrained=True, freeze_backbone=True, tiles_per_side_values=[1, 4, 7, 10], num_tile_permutations=3, epochs=10, learning_rate=0.0003, optimizer='adamw', weight_decay=0.0001, use_amp=True, seed=42, max_threads=4, using_google_colab=True, plot_samples=False, stage_colab_data_to_local_disk=True, colab_local_data_dir='/content/MLDS_Final_Project/data/dogs-vs-cats/train', profile_perform

In [7]:
%%time
config.data_dir = stage_configured_colab_data_dir(config)
print(f'Active data_dir: {config.data_dir}')


Starting Colab dataset staging.
  Source data directory: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train
  Expected source ZIP: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train.zip
  Local ZIP destination: /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip
  Local extraction directory: /content/MLDS_Final_Project/data/dogs-vs-cats/train
  Source directory labeled images: 25000
  Source ZIP labeled images: 25000
  Existing local labeled images: 0
Copying dataset ZIP from Google Drive to local Colab disk: /content/drive/MyDrive/MLDS_Final_Project/data/dogs-vs-cats/train.zip -> /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip (547.8 MiB)
Finished copying dataset ZIP in 7.73s.
Extracting 25000 labeled images from /content/MLDS_Final_Project/data/dogs-vs-cats/train.zip into /content/MLDS_Final_Project/data/dogs-vs-cats/train.
Finished extracting Colab dataset: 25000 files extracted in 3.75s; 25000 labeled images now available locally.
Finished 

In [8]:

METRIC_LABELS = {
    'global_tile_displacement': 'Global displacement',
    'adjacency_destruction_hardness': 'Adjacency destruction',
    'spatial_permutation_entropy': 'Spatial permutation entropy',
    'combined_hardness_score': 'Combined hardness',
}
CORRELATION_METRIC_LABELS = {
    'global_tile_displacement': 'Global tile displacement (higher = more moved)',
    'adjacency_destruction_hardness': 'Adjacency destruction (higher = fewer neighbors preserved)',
    'spatial_permutation_entropy': 'Spatial permutation entropy (higher = more varied movement)',
    'combined_hardness_score': 'Combined hardness score (weighted summary)',
}
ANALYSIS_SCOPE_LABELS = {
    'one_baseline_row': 'Primary: one clean baseline row',
    'all_rows': 'All saved rows, including repeated clean baselines',
    'non_baseline_rows': 'Only permuted rows, clean baseline removed',
}
ANALYSIS_SCOPE_ORDER = list(ANALYSIS_SCOPE_LABELS)
METRIC_COLUMNS = list(METRIC_LABELS)


def _part3_table_style(styler):
    return (
        styler.hide(axis='index')
        .set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left'), ('white-space', 'normal'), ('max-width', '180px')]},
            {'selector': 'td', 'props': [('text-align', 'left'), ('white-space', 'normal'), ('max-width', '220px')]},
        ])
    )


def display_part3_metrics_table(metrics: pd.DataFrame, max_rows: int = 12):
    print(f'Displaying {min(len(metrics), max_rows)} of {len(metrics)} tile-order metric rows.')
    table = metrics.copy()
    table['_reordered_first'] = ~(table['tiles_per_side'].isna() | (table['tiles_per_side'] == 1) | (table['tile_permutation_id'] == 0))
    table = table.sort_values(['_reordered_first', 'tiles_per_side', 'tile_permutation_id'], ascending=[False, True, True])
    table = table.drop(columns=['_reordered_first'])
    table = table.rename(columns=METRIC_LABELS | {
        'tiles_per_side': 'Grid side',
        'num_tiles': 'Total tiles',
        'tile_permutation_id': 'Tile order ID',
        'tile_permutation_name': 'Difficulty label',
        'tile_permutation_seed': 'Seed',
    })
    display(_part3_table_style(table.head(max_rows).style.format(precision=3)))


def display_part3_joined_table(joined: pd.DataFrame, max_rows: int = 12):
    print(f'Displaying {min(len(joined), max_rows)} of {len(joined)} metric-accuracy joined rows.')
    columns = ['tiles_per_side', 'num_tiles', 'tile_permutation_id', 'tile_permutation_name', 'best_val_accuracy', *METRIC_COLUMNS]
    table = joined.loc[:, [column for column in columns if column in joined.columns]].copy()
    table['_reordered_first'] = ~(table['tiles_per_side'].isna() | (table['tiles_per_side'] == 1) | (table['tile_permutation_id'] == 0))
    table = table.sort_values(['_reordered_first', 'tiles_per_side', 'tile_permutation_id'], ascending=[False, True, True])
    table = table.drop(columns=['_reordered_first'])
    table = table.rename(columns=METRIC_LABELS | {
        'tiles_per_side': 'Grid side',
        'num_tiles': 'Total tiles',
        'tile_permutation_id': 'Tile order ID',
        'tile_permutation_name': 'Difficulty label',
        'best_val_accuracy': 'Best validation accuracy',
    })
    display(_part3_table_style(table.head(max_rows).style.format(precision=3)))


def display_part3_correlations_table(correlations: pd.DataFrame):
    table = correlations.copy()
    table['_scope_order'] = pd.Categorical(
        table['analysis_scope'],
        categories=ANALYSIS_SCOPE_ORDER,
        ordered=True,
    )
    table['analysis_scope'] = table['analysis_scope'].replace(ANALYSIS_SCOPE_LABELS)
    table['metric'] = table['metric'].replace(CORRELATION_METRIC_LABELS)
    table = table.sort_values(['_scope_order', 'metric']).drop(columns=['_scope_order'])
    table = table.rename(columns={
        'group': 'Model',
        'analysis_scope': 'Rows included',
        'metric': 'Hardness metric',
        'pearson': 'Pearson r: linear link with accuracy',
        'spearman': 'Spearman rho: rank link with accuracy',
        'n': 'Rows used',
    })
    display(_part3_table_style(table.style.format({
        'Pearson r: linear link with accuracy': '{:.3f}',
        'Spearman rho: rank link with accuracy': '{:.3f}',
    })))


def print_part3_correlation_summary(correlations: pd.DataFrame):
    primary = correlations[correlations['analysis_scope'] == 'one_baseline_row'].copy()
    if primary.empty:
        print('Primary correlation scope was not found in the saved Part 3 results.')
        return

    print('Correlation reading guide: negative values mean harder permutations are associated with lower validation accuracy.')
    print('Primary scope keeps one clean 1x1 baseline row, so the repeated easy/medium/hard clean baseline does not get triple weight.')
    print(f"Primary rows used: {int(primary['n'].max())}")

    ranked = primary.dropna(subset=['pearson', 'spearman']).copy()
    if ranked.empty:
        print('No defined primary correlations were available; check for constant metric or accuracy values.')
        return

    ranked['_abs_spearman'] = ranked['spearman'].abs()
    strongest = ranked.sort_values('_abs_spearman', ascending=False).iloc[0]
    metric_name = CORRELATION_METRIC_LABELS.get(str(strongest['metric']), str(strongest['metric']))
    print(
        'Strongest primary rank association: '
        f"{metric_name} (Spearman rho={strongest['spearman']:.3f}, Pearson r={strongest['pearson']:.3f})."
    )


def _deserialize_tile_permutation(raw_tile_permutation, tiles_per_side):
    if raw_tile_permutation is None:
        return None
    if isinstance(raw_tile_permutation, float) and pd.isna(raw_tile_permutation):
        return None
    if isinstance(raw_tile_permutation, str):
        raw_tile_permutation = json.loads(raw_tile_permutation)
    return tile_permutation_from_jsonable(raw_tile_permutation, tiles_per_side)


def _deduplicate_part3_baseline_rows(tile_permutations: pd.DataFrame) -> pd.DataFrame:
    is_baseline = tile_permutations['tiles_per_side'].isna()
    if 'tile_permutation' in tile_permutations.columns:
        is_baseline = is_baseline | tile_permutations['tile_permutation'].isna()
    baseline_rows = tile_permutations[is_baseline].head(1)
    permutation_rows = tile_permutations[~is_baseline]
    return pd.concat([baseline_rows, permutation_rows], ignore_index=True)


def _sorted_part3_tile_permutations(tile_permutation_csv: Path, config: Part3ExperimentConfig) -> pd.DataFrame:
    tile_permutations = load_or_build_part1_tile_permutations(
        tile_permutation_csv=str(tile_permutation_csv),
        tiles_per_side_values=config.tiles_per_side_values,
        num_tile_permutations=config.num_tile_permutations,
        seed=config.seed,
    ).copy()
    tile_permutations = _deduplicate_part3_baseline_rows(tile_permutations)
    tile_permutations['_tiles_sort'] = tile_permutations['tiles_per_side'].fillna(0).astype(int)
    tile_permutations = tile_permutations.sort_values(['_tiles_sort', 'tile_permutation_id'])
    return tile_permutations.drop(columns=['_tiles_sort']).reset_index(drop=True)


def _load_example_tensor(image_path: Path, image_size: int, tiles_per_side: int):
    tile_compatible_size = make_tile_compatible_image_size(image_size, tiles_per_side)
    with PILImage.open(image_path) as image:
        resized = image.convert('RGB').resize(
            (tile_compatible_size, tile_compatible_size),
            PILImage.Resampling.BILINEAR,
        )
    return PILToFloatTensor()(resized)


def _tensor_to_image_array(image_tensor):
    return image_tensor.permute(1, 2, 0).clamp(0.0, 1.0).cpu().numpy()


def build_part3_example_image_metrics(
    validation_samples,
    tile_permutation_csv: Path,
    config: Part3ExperimentConfig,
) -> tuple[Path, pd.DataFrame]:
    if not validation_samples:
        raise ValueError('validation_samples must contain at least one image')

    image_path = Path(sorted(validation_samples, key=lambda sample: str(sample[0]))[0][0])
    rows = []
    for _, row in _sorted_part3_tile_permutations(tile_permutation_csv, config).iterrows():
        raw_tiles_per_side = row['tiles_per_side']
        tiles_per_side = None if pd.isna(raw_tiles_per_side) else int(raw_tiles_per_side)
        effective_tiles_per_side = 1 if tiles_per_side is None else tiles_per_side
        tile_permutation = _deserialize_tile_permutation(row['tile_permutation'], tiles_per_side)

        image_tensor = _load_example_tensor(image_path, config.image_size, effective_tiles_per_side)
        if tile_permutation is None:
            displayed_tensor = image_tensor
        else:
            displayed_tensor = apply_tile_permutation(image_tensor, tile_permutation)

        rows.append({
            'tiles_per_side': tiles_per_side,
            'grid': 'baseline' if tiles_per_side is None else f'{tiles_per_side}x{tiles_per_side}',
            'num_tiles': 1 if tiles_per_side is None else tiles_per_side * tiles_per_side,
            'tile_permutation_id': int(row['tile_permutation_id']),
            'tile_permutation_name': 'no permutation' if tile_permutation is None else row.get('tile_permutation_name'),
            'image_array': _tensor_to_image_array(displayed_tensor),
            'global_tile_displacement': compute_global_displacement(tile_permutation, tiles_per_side),
            'adjacency_destruction_hardness': compute_adjacency_destruction_hardness(
                tile_permutation,
                tiles_per_side,
            ),
            'spatial_permutation_entropy': compute_spatial_permutation_entropy(
                tile_permutation,
                tiles_per_side,
            ),
        })

    examples = pd.DataFrame(rows)

    examples['combined_hardness_score'] = [
        compute_combined_hardness(
            adjacency_destruction_hardness=float(row['adjacency_destruction_hardness']),
            spatial_permutation_entropy=float(row['spatial_permutation_entropy']),
            global_tile_displacement=float(row['global_tile_displacement']),
            weight_adj=config.weight_adj,
            weight_entropy=config.weight_entropy,
            weight_dist=config.weight_dist,
        )
        for _, row in examples.iterrows()
    ]
    return image_path, examples


def display_part3_example_image_metrics(image_path: Path, examples: pd.DataFrame):
    row_count = len(examples)
    fig, axes = plt.subplots(
        row_count,
        2,
        figsize=(9, max(2.1, 2.1 * row_count)),
        gridspec_kw={'width_ratios': [1.0, 1.35]},
        squeeze=False,
    )
    fig.suptitle(f'Deterministic validation image: {image_path.name}', y=0.995)
    for axis_row, (_, row) in zip(axes, examples.iterrows()):
        image_axis, text_axis = axis_row
        image_axis.imshow(row['image_array'])
        image_axis.set_title(f"Grid {row['grid']} | {row.get('tile_permutation_name', 'order')}")
        image_axis.axis('off')

        text_axis.axis('off')
        text_axis.text(
            0.0,
            0.5,
            '\n'.join([
                f"Adjacency destruction: {row['adjacency_destruction_hardness']:.3f}",
                f"Global displacement: {row['global_tile_displacement']:.3f}",
                f"Spatial permutation entropy: {row['spatial_permutation_entropy']:.3f}",
                f"Combined hardness: {row['combined_hardness_score']:.3f}",
            ]),
            va='center',
            ha='left',
            fontsize=11,
        )
    fig.tight_layout(rect=[0, 0, 1, 0.985])
    plt.show()

    table_columns = [
        'grid',
        'tile_permutation_id',
        'tile_permutation_name',
        'adjacency_destruction_hardness',
        'global_tile_displacement',
        'spatial_permutation_entropy',
        'combined_hardness_score',
    ]
    display(_part3_table_style(examples[table_columns].style.format(precision=3)))


## Data Loading
Load existing Part 1 results filtered to the configured Part 3 model.

In [9]:

part1_model_results = load_part1_model_results(str(part1_results_csv), config.model_name)
print(f'Part 1 {config.model_name} result rows: {len(part1_model_results)}')
display(
    part1_model_results[
        ['tiles_per_side', 'num_tiles', 'tile_permutation_id', 'best_val_accuracy']
    ].sort_values(['tiles_per_side', 'tile_permutation_id']).head(12).style.format(precision=3)
)


Part 1 resnet18 result rows: 12


,tiles_per_side,num_tiles,tile_permutation_id,best_val_accuracy
3,4.000,16,1,0.974
4,4.000,16,2,0.943
5,4.000,16,3,0.910
6,7.000,49,1,0.976
7,7.000,49,2,0.927
8,7.000,49,3,0.848
9,10.000,100,1,0.971
10,10.000,100,2,0.860
11,10.000,100,3,0.805
0,nan,1,1,0.979


## Compute Hardness Metrics
Compute tile permutation-only metrics, join them with the configured model accuracy, and save outputs.

In [ ]:

_, validation_samples, _ = load_experiment_samples(config, seed=config.seed)

analysis_results = run_part3_hardness_analysis(
    results_dir=str(results_dir),
    figures_dir=str(figures_dir),
    part1_results_csv=str(part1_results_csv),
    tile_permutation_csv=str(tile_permutation_csv),
    tiles_per_side_values=config.tiles_per_side_values,
    num_tile_permutations=config.num_tile_permutations,
    seed=config.seed,
    validation_samples=validation_samples,
    image_size=config.image_size,
    weight_adj=config.weight_adj,
    weight_entropy=config.weight_entropy,
    weight_dist=config.weight_dist,
    model_name=config.model_name,
    verbose=True,
    show_progress=True,
)

display_part3_metrics_table(analysis_results['metrics'])
display_part3_joined_table(analysis_results['joined'])


## Example Images and Image-Specific Scores
Display one deterministic validation image for each available tile permutation, with image-specific metric calculations.

In [ ]:
example_image_path, example_image_metrics = build_part3_example_image_metrics(
    validation_samples=validation_samples,
    tile_permutation_csv=tile_permutation_csv,
    config=config,
)
display_part3_example_image_metrics(example_image_path, example_image_metrics)


## Correlations
Display Pearson and Spearman correlations between each hardness metric and the configured model validation accuracy. The primary scope is `one_baseline_row`, which keeps one no-permutation baseline instead of triple-weighting the repeated 1x1 easy/medium/hard rows; `all_rows` is retained for historical comparability, and `non_baseline_rows` is a robustness check.

In [ ]:

saved_results = load_part3_results(str(results_dir))
print_part3_correlation_summary(saved_results['correlations'])
display_part3_correlations_table(saved_results['correlations'])

if saved_results['correlations'][['pearson', 'spearman']].isna().any().any():
    print(
        'NaN means the correlation is undefined because the selected accuracy '
        'or metric values are constant in the joined Part 3 data.'
    )


## Metric Plots
Display saved metric-vs-accuracy plots.

In [ ]:

output_paths = part3_output_paths(str(results_dir), str(figures_dir))
plot_paths = [figure_path for figure_path in output_paths['plots'] if '_3d' not in Path(figure_path).stem]
print(f'Displaying {len(plot_paths)} saved 2D Part 3 metric plots. 3D plots are intentionally hidden.')
for figure_path in plot_paths:
    display(Image(filename=figure_path))
if not plot_paths:
    print('No 2D Part 3 plots found yet.')


In [14]:
# Export this saved notebook to PDF. Save the notebook before running this cell,
# because nbconvert reads the on-disk .ipynb file rather than unsaved editor state.
import importlib

import src.utils.notebook_setup as notebook_setup

notebook_setup = importlib.reload(notebook_setup)
notebook_path = ROOT / 'src' / 'notebooks' / 'part3_solution.ipynb'
export_dir = ROOT / 'outputs' / 'notebooks'
notebook_setup.export_notebook_to_pdf(notebook_path, export_dir)


FileNotFoundError: [Errno 2] No such file or directory: 'kpsewhich'